# Ansatz A - regelbasierte Transformation

**Concept:** Strict if-then rules and naming convention mapping. No LLM calls, no probabilistic decisions—all mappings are deterministic and traceable.

**Design decisions:**
- **Name-based mapping** with ID fallback (as decided in the discussion)

- **Mapping rules centralized in `COMPONENT_MAP`** — each component gets its own entry with props, value, and variant rules

- **Explicit skip properties** — all pure Figma visualization props (`State`, `Hover`, `Focus`, etc.) are filtered out

- **Sub-instances with an `_` prefix** are not rendered independently (they are children of components such as `password`/`inputtext`)

- **Wrapper frames (Column/Row/content)** are translated into `<div>` with Tailwind classes


In [21]:
import re
import json
import time
from typing import Any
from pathlib import Path
from dataclasses import dataclass, field
from collections import Counter, defaultdict

In [22]:
INPUT_DIR = 'dataset/figma-data/cleaned'
OUTPUT_DIR = 'dataset/storybook/src/stories'

## 1. Mapping-Configuration

In [23]:
UNIVERSAL_SKIP = {'State', 'Hover', 'Focus', 'Pressed', 'Active'}

# Primitiv PrimeVue-Components
COMPONENT_MAP: dict[str, dict] = {
    # === Button ===
    'button': {
        'primevue': 'Button',
        'props': {
            'Text#4293:477': {
                'target': 'label',
                'type': 'text'
            },
            'Severity': {
                'target': 'severity',
                'type': 'enum',
                'value_map': {
                    'Primary': None,  # Default — Omit prop
                    'Plain': None,
                    'Secondary': 'secondary',
                    'Success': 'success',
                    'Info': 'info',
                    'Warning': 'warn',
                    'Help': 'help',
                    'Danger': 'danger',
                    'Contrast': 'contrast',
                },
            },
            'Disabled': {
                'target': 'disabled',
                'type': 'boolean',
                'omit_when': False
            },
            '⥰ Rounded': {
                'target': 'rounded',
                'type': 'boolean',
                'omit_when': False
            },
        },
        'variant_resolver': {
            'inputs': [
                '🔲 Outlined',
                '🔤 Text',
                'Link',
                '⬆️ Raised'
            ],
            'target': 'variant',
            'rules': [
                {
                    'when': {
                        '🔲 Outlined': 'True'
                    },
                    'result': 'outlined'
                },
                {
                    'when': {
                        '🔤 Text': 'True'
                    },
                    'result': 'text'
                },
                {
                    'when': {
                        'Link': 'True'
                    },
                    'result': 'link'
                },
            ],
        },
        'icon_only_prop': 'Icon Only',
        'skip': UNIVERSAL_SKIP | {
            'Show Right Icon#1644:1387',
            'Show Left Icon#1644:0',
            'Right Icon#1644:4161',
            'Left Icon#1644:2774',
            'Icon#1690:0',
            'Icon Only', # Not a prop; evaluated only via `icon_only_prop`
            '⬆️ Raised', # “raised” no longer exists in PrimeVue v4
        },
        'slot_strategy': 'drop',  # Children are Figma's built-in icon components
    },

    # === InputText ===
    'inputtext': {
        'primevue': 'InputText',
        'props': {
            '↳ Float Label#4275:152': {
                'target': 'placeholder',
                'type': 'text'
            },
            '🚫 Disabled': {
                'target': 'disabled',
                'type': 'boolean',
                'omit_when': False
            },
            '❌ Invalid': {
                'target': 'invalid',
                'type': 'boolean',
                'omit_when': False
            },
            '🟦 Filled': {
                'target': 'variant',
                'type': 'enum',
                'value_map': {
                    'False': None,
                    'True': 'filled'
                }
            },
            '🤏 Size': {
                'target': 'size',
                'type': 'enum',
                'value_map': {
                    'Normal': None,
                    'Small': 'small',
                    'Large': 'large'
                }
            },
        },
        'skip': UNIVERSAL_SKIP | {
            '⚙️ State',
            'ℹ️ Show Helper#1729:1731',
            '↳ Helper Text#1729:4039',
            '↳ Label#5662:83',
            '🏷️ Show Label#5662:50',
            '➡️ Show Right Icon#1729:3462',
            '↳ Right Icon#1729:577',
            '⬅️ Show Left Icon#1729:2885',
            '↳ Left Icon#1729:0',
            '👁️ Show Text#1729:2308',
            '▭ Ifta Label',
            '🏷️ Float Label',
            '↳ Float Label Variant',
        },
        'slot_strategy': 'drop',  # Internal structure is rendered by PrimeVue
        'extra_attrs': {'v-model': '_state.{node_id}'},  # 2-way binding placeholder
    },

    # === Password ===
    'password': {
        'primevue': 'Password',
        'props': {
            'Toggle Mask': {
                'target': 'toggleMask',
                'type': 'boolean',
                'omit_when': False
            },
        },
        'skip': UNIVERSAL_SKIP | {'Password Visible'},
        'slot_strategy': 'drop',
        'extra_attrs': {'v-model': '_state.{node_id}'},
        # Placeholder comes from the internal _inputtext-content sub-instance
        'inherit_placeholder_from_sub': '_inputtext-content',
    },

    # === InputNumber ===
    'inputnumber': {
        'primevue': 'InputNumber',
        'props': {
            'Type': {
                'target': 'buttonLayout', 'type': 'enum',
                'value_map': {
                    'Vertical': 'vertical',
                    'Horizontal': 'horizontal',
                    'Horizontal with Step': 'horizontal',
                    'Stacked': 'stacked',
                    'Default': None
                }
            },
        },
        'skip': UNIVERSAL_SKIP,
        'slot_strategy': 'drop',
        'extra_attrs': {
            'v-model': '_state.{node_id}',
            'showButtons': True
        },
        'inherit_placeholder_from_sub': '_inputtext-content',
    },

    # === Avatar ===
    'avatar': {
        'primevue': 'Avatar',
        'props': {
            'Text#4271:0': {
                'target': 'label',
                'type': 'text'
            },
            'Size': {
                'target': 'size',
                'type': 'enum',
                'value_map': {
                    'Normal': None,
                    'Large': 'large',
                    'X-Large': 'xlarge'
                }
            },
            'Circle': {
                'target': 'shape',
                'type': 'enum',
                'value_map': {
                    'True': 'circle',
                    'False': None
                }
            },
            'Type': {
                'target': '_type',
                'type': 'enum',
                'value_map': {
                    'Label': None,
                    'Icon': None,
                    'Image': None
                }
            },
        },
        'skip': UNIVERSAL_SKIP,
        'slot_strategy': 'drop',  # Text is set via the label property; badge is set separately (see below)
        'overlay_badge_child': True,  # If Show Badge=True → wrap with OverlayBadge
    },

    # === OverlayBadge ===
    'overlaybadge': {
        'primevue': 'OverlayBadge',
        'props': {
            'Text#4272:0': {
                'target': 'value',
                'type': 'text'
            },
            'Severity': {
                'target': 'severity',
                'type': 'enum',
                'value_map': {
                    'Primary': None,
                    'Secondary': 'secondary',
                    'Success': 'success',
                    'Info': 'info',
                    'Warning': 'warn',
                    'Danger': 'danger',
                    'Contrast': 'contrast'
                },
            },
            'Size': {
                'target': 'size',
                'type': 'enum',
                'value_map': {
                    'Default': None,
                    'Large': 'large',
                    'X-Large': 'xlarge'
                }
            },
        },
        'skip': UNIVERSAL_SKIP | {'Circle'},
        'slot_strategy': 'drop',
    },

    # === Slider ===
    'slider': {
        'primevue': 'Slider',
        'props': {
            'Direction': {
                'target': 'orientation',
                'type': 'enum',
                'value_map': {
                    'Horizontal': None,
                    'Vertical': 'vertical'
                }
            },
            'Disabled': {
                'target': 'disabled',
                'type': 'boolean',
                'omit_when': False
            },
            'Range': {
                'target': 'range',
                'type': 'boolean',
                'omit_when': False
            },
        },
        'skip': UNIVERSAL_SKIP | {
            'Input',
            'Slide'
        },
        'slot_strategy': 'drop',
        'extra_attrs': {'v-model': '_state.{node_id}'},
    },

    # === Checkbox ===
    'checkbox': {
        'primevue': 'Checkbox',
        'props': {
            'Disabled': {
                'target': 'disabled',
                'type': 'boolean',
                'omit_when': False
            },
            'Filled': {
                'target': 'variant',
                'type': 'enum',
                 'value_map': {
                     'False': None,
                     'True': 'filled'
                 }
            },
            'Size': {
                'target': 'size',
                'type': 'enum',
                'value_map': {
                    'Normal': None,
                    'Small': 'small',
                    'Large': 'large'
                }
            },
        },
        'skip': UNIVERSAL_SKIP | {
            'Focus', 'Hover',
            'Selected',
            'Label#4275:102',
            'Show Label#1844:0',
        },
        'slot_strategy': 'drop',
        'extra_attrs': {
            'v-model': '_state.{node_id}',
            'binary': True
        },
    },

    # === RadioButton ===
    'radiobutton': {
        'primevue': 'RadioButton',
        'props': {
            'Disabled': {
                'target': 'disabled',
                'type': 'boolean',
                'omit_when': False
            },
            'Filled': {
                'target': 'variant',
                'type': 'enum',
                'value_map': {
                    'False': None,
                    'True': 'filled'
                }
            },
            'Size': {
                'target': 'size',
                'type': 'enum',
                'value_map': {
                    'Normal': None,
                    'Small': 'small',
                    'Large': 'large'
                }
            },
        },
        'skip': UNIVERSAL_SKIP | {
            'Selected',
            'Label#4283:5',
            'Show Label#1850:29',
        },
        'slot_strategy': 'drop',
        'extra_attrs': {
            'v-model': '_state.{node_id}',
            'value': 'option1'
        },
    },

    # === ToggleSwitch ===
    'toggleswitch': {
        'primevue': 'ToggleSwitch',
        'props': {
            'Disabled': {
                'target': 'disabled',
                'type': 'boolean',
                'omit_when': False
            },
            'Invalid': {
                'target': 'invalid',
                'type': 'boolean',
                'omit_when': False
            },
        },
        'skip': UNIVERSAL_SKIP | {
            'Focus',
            'Checked'
        },
        'slot_strategy': 'drop',
        'extra_attrs': {'v-model': '_state.{node_id}'},
    },

    # === Textarea ===
    'textarea': {
        'primevue': 'Textarea',
        'props': {
            '↳ Float Label#4275:152': {
                'target': 'placeholder',
                'type': 'text'
            },
            '🚫 Disabled': {
                'target': 'disabled',
                'type': 'boolean',
                'omit_when': False
            },
            '❌ Invalid': {
                'target': 'invalid',
                'type': 'boolean',
                'omit_when': False
            },
            '🟦 Filled': {
                'target': 'variant',
                'type': 'enum',
                'value_map': {
                    'False': None,
                    'True': 'filled'
                }
            },
            '🤏 Size': {
                'target': 'size',
                'type': 'enum',
                'value_map': {
                    'Normal': None,
                    'Small': 'small',
                    'Large': 'large'
                }
            },
        },
        'skip': UNIVERSAL_SKIP | {
            '⚙️ State',
            '👁️ Show Text#1729:2308',
            '▭ Ifta Label',
            '🏷️ Float Label',
            '↳ Float Label Variant',
        },
        'slot_strategy': 'drop',
        'extra_attrs': {
            'v-model': '_state.{node_id}',
            'rows': '4'
        },
        'inherit_placeholder_from_sub': '_textarea-content',
    },

    # === Tag ===
    'tag': {
        'primevue': 'Tag',
        'props': {
            'Text#4272:22': {
                'target': 'value',
                'type': 'text'
            },
            'Severity': {
                'target': 'severity',
                'type': 'enum',
                'value_map': {
                    'Primary': None,
                    'Secondary': 'secondary',
                    'Success': 'success',
                    'Info': 'info',
                    'Warning': 'warn', 'Warn': 'warn',
                    'Danger': 'danger',
                    'Contrast': 'contrast',
                },
            },
            'Rounded': {
                'target': 'rounded',
                'type': 'boolean',
                'omit_when': False
            },
        },
        'skip': UNIVERSAL_SKIP | {
            'Icon#2143:55',
            'Show Icon#2143:34'
        },
        'slot_strategy': 'drop',
    },

    # === Divider ===
    'divider': {
        'primevue': 'Divider',
        'props': {
            'Direction': {
                'target': 'layout',
                'type': 'enum',
                'value_map': {
                    'Horizontal': None,
                    'Vertical': 'vertical'
                }
            },
            'Type': {
                'target': 'type',
                'type': 'enum',
                'value_map': {
                    'Solid': None,
                    'Dashed': 'dashed',
                    'Dotted': 'dotted'
                }
            },
            'Align': {
                'target': 'align',
                'type': 'enum',
                'value_map': {
                    'Center': None,
                    'Left': 'left',
                    'Right': 'right'
                }
            },
        },
        'skip': UNIVERSAL_SKIP | {'Content'},
        'slot_strategy': 'drop',
    },

    # === ProgressBar ===
    'progressbar': {
        'primevue': 'ProgressBar',
        'props': {
            'Value': {
                'target': 'showValue',
                'type': 'enum',
                'value_map': {
                    'True': None,
                    'False': 'false'
                },
            },
            'Type': {
                'target': 'mode',
                'type': 'enum',
                'value_map': {
                    'Basic': None,
                    'Indeterminate': 'indeterminate'
                },
            },
            'Text#4271:24': {
                'target': '_pct_raw',
                'type': 'text'
            },
        },
        'skip': UNIVERSAL_SKIP,
        'slot_strategy': 'drop',
        'parse_value_from': 'Text#4271:24',
    },

    # === DatePicker ===
    'datepicker': {
        'primevue': 'DatePicker',
        'props': {
            'Picker Type': {
                'target': 'selectionMode',
                'type': 'enum',
                'value_map': {
                    'Single': None,
                    'Multiple': 'multiple',
                    'Range': 'range'
                }
            },
            'Type': {
                'target': 'view',
                'type': 'enum',
                'value_map': {
                    'Date': None,
                    'Month': 'month',
                    'Year': 'year'
                }
            },
            'Show Bar#2070:0': {
                'target': 'showButtonBar',
                'type': 'boolean',
                'omit_when': False
            },
            'Show Week#1879:0': {
                'target': 'showWeek',
                'type': 'boolean',
                'omit_when': False
            },
        },
        'skip': UNIVERSAL_SKIP,
        'slot_strategy': 'drop',
        'extra_attrs': {'v-model': '_state.{node_id}'},
    },

    # === Menu ===
    'menu': {
        'primevue': 'Menu',
        'props': {},  # `model` is an array — it is set as a placeholder below via `extra_attrs`
        'skip': UNIVERSAL_SKIP | {
            'Type',
            'Show Button#4683:0',
            'Show Menu#2125:5'
        },
        'slot_strategy': 'drop',
        'extra_attrs': {':model': '_menuItems'},
    },

    # === Skeleton ===
    'skeleton': {
        'primevue': 'Skeleton',
        'props': {},
        'skip': UNIVERSAL_SKIP,
        'slot_strategy': 'drop',
    },
}

ICON_SKIP_INSTANCES = {
    'chevron-down',
    'chevron-up',
    'chevron-right',
    'chevron-left',
    'times',
    'ellipsis-h',
    'search',
    'check',
    'plus',
    'minus',
    'trash',
    'pen-to-square',
    'clone',
    'flag',
    'inbox',
    'folder',
    'user',
    'shield',
    'angle-left',
    'angle-right',
    'cart-plus',
    'calendar',
    'clock',
    'arrow-up-right',
}

print(f'Components in Mapping: {len(COMPONENT_MAP)}')
for name in sorted(COMPONENT_MAP.keys()):
    pv = COMPONENT_MAP[name]["primevue"]
    print(f'  {name:25s} -> <{pv}>')

# Alias: ‘calendar’ → DatePicker (former Figma component name)
COMPONENT_MAP['calendar'] = COMPONENT_MAP['datepicker']

TRANSPARENT_FRAMES = {'screen'}

FRAME_MAP: dict[str, dict] = {
    # === Card ===
    'card': {
        'primevue': 'Card',
        'strategy': 'card',
        # body-Frame is transparent; caption/content/footer are mapped with slots
        'slot_map': {
            'header': 'header',
            'content': 'content',
            'footer': 'footer'
        },
    },

    # === Dialog ===
    'dialog': {
        'primevue': 'Dialog',
        'strategy': 'slot_map',
        'slot_map': {
            'header':  'header',     # → <template #header>
            'content': None,         # → Default-Slot
            'footer':  'footer',     # → <template #footer>
            'actions': 'header',     # Close-Button → ins Header
        },
        'extra_attrs': {
            ':modal': 'true',
            'v-model:visible': '_state.{node_id}Visible'
        },
    },

    # === Tabs ===
    'tabs': {
        'primevue': 'Tabs',
        'strategy': 'tabs',
    },

    # === DataTable ===
    'datatable': {
        'primevue': 'DataTable',
        'strategy': 'datatable',
    },

    # === Select ===
    'select': {
        'primevue': 'Select',
        'strategy': 'select',
    },

    # === Popover ===
    'popover': {
        'primevue': 'Popover',
        'strategy': 'popover',
    },

    # === Breadcrumb ===
    'breadcrumb': {
        'primevue': 'Breadcrumb',
        'strategy': 'breadcrumb',
    },

    # === Accordion ===
    'accordion': {
        'primevue': 'Accordion',
        'strategy': 'accordion',
    },
}

print(f'Frame-based Components in Mapping: {len(FRAME_MAP)}')
for name in sorted(FRAME_MAP.keys()):
    pv = FRAME_MAP[name]["primevue"]
    print(f'  {name:25s} -> <{pv}>')


Components in Mapping: 17
  avatar                    -> <Avatar>
  button                    -> <Button>
  checkbox                  -> <Checkbox>
  datepicker                -> <DatePicker>
  divider                   -> <Divider>
  inputnumber               -> <InputNumber>
  inputtext                 -> <InputText>
  menu                      -> <Menu>
  overlaybadge              -> <OverlayBadge>
  password                  -> <Password>
  progressbar               -> <ProgressBar>
  radiobutton               -> <RadioButton>
  skeleton                  -> <Skeleton>
  slider                    -> <Slider>
  tag                       -> <Tag>
  textarea                  -> <Textarea>
  toggleswitch              -> <ToggleSwitch>
Frame-based Components in Mapping: 8
  accordion                 -> <Accordion>
  breadcrumb                -> <Breadcrumb>
  card                      -> <Card>
  datatable                 -> <DataTable>
  dialog                    -> <Dialog>
  popover  

In [24]:
_metrics: dict[str, int] = {}

def _reset_metrics():
    global _metrics
    _metrics = {
        'instances_mapped': 0,
        'instances_unmapped': 0,
        'frames_compound': 0,
        'frames_fallback': 0,
    }

def _ast_depth(node) -> int:
    if not isinstance(node, UINode) or not node.children:
        return 1
    return 1 + max(
        (_ast_depth(c) for c in node.children if isinstance(c, UINode)),
        default=0,
    )

## 2. AST-Definition

In [25]:
@dataclass
class UINode:
    tag: str  # 'Button', 'div', etc.
    props: dict = field(default_factory=dict)  # static Props
    dynamic_props: dict = field(default_factory=dict)  # with `:`-Prefix
    classes: list[str] = field(default_factory=list)  # Tailwind-Classes
    children: list = field(default_factory=list)  # more UINodes or Strings
    slot: str | None = None  # if a child of a component slot
    is_component: bool = False  # PrimeVue-Component vs. HTML-Element
    self_closing: bool = False
    figma_id: str | None = None  # Tracking for Debugging
    figma_name: str | None = None

    def __repr__(self):
        return f'UINode({self.tag}, props={list(self.props)}, children={len(self.children)})'

## 3. Helper: Apply Property-Mapping

In [26]:
def _extract_value(prop_entry: Any) -> Any:
    if isinstance(prop_entry, dict) and 'value' in prop_entry:
        return prop_entry['value']

    return prop_entry


def _convert_value(raw: Any, prop_type: str) -> Any:
    if prop_type == 'boolean':
        if isinstance(raw, bool):
            return raw
        if isinstance(raw, str):
            return raw.lower() == 'true'

    if prop_type == 'number':
        try:
            return float(raw)
        except (TypeError, ValueError):
            return None

    return raw

In [27]:
def apply_property_rules(figma_props: dict, spec: dict) -> tuple[dict, dict]:
    """Mapping component through mapping rules

    Returns:
        (static_props, dynamic_props) — separated by string- and boolean-values
    """
    static = {}
    dynamic = {}
    skip = spec.get('skip', set())
    rules = spec.get('props', {})

    # 1) Simple 1:1-Mappings
    for figma_key, rule in rules.items():
        if figma_key not in figma_props:
            continue

        raw = _extract_value(figma_props[figma_key])
        prop_type = rule.get('type', 'text')
        target = rule['target']

        # Value-Mapping (example: 'Primary' -> None)
        if 'value_map' in rule:
            if raw not in rule['value_map']:
                continue
            mapped = rule['value_map'][raw]
            if mapped is None:
                continue  # Default-Value → omit
            value = mapped
        else:
            value = _convert_value(raw, prop_type)

        if value is None:
            continue
        if 'omit_when' in rule and value == rule['omit_when']:
            continue

        # Booleans / Numbers over dynamic-binding return
        if prop_type == 'boolean':
            dynamic[target] = 'true' if value else 'false'
        elif prop_type == 'number':
            dynamic[target] = str(value)
        else:
            static[target] = str(value)

    # 2) Variant-Resolver: several Booleans -> one Enum-Prop
    vr = spec.get('variant_resolver')
    if vr:
        for rule in vr['rules']:
            match = all(_extract_value(figma_props.get(k)) == v
                        for k, v in rule['when'].items())
            if match:
                static[vr['target']] = rule['result']
                break

    return static, dynamic

## 4. Layout-Engine: FRAME → Tailwind

In [28]:
TAILWIND_SPACING = {
    0: '0', 1: '0.5', 2: '0.5', 4: '1', 6: '1.5', 8: '2', 10: '2.5', 12: '3',
    14: '3.5', 16: '4', 20: '5', 24: '6', 28: '7', 32: '8', 36: '9', 40: '10',
    44: '11', 48: '12', 56: '14', 64: '16',
}


def _spacing_class(prefix: str, px: float | int | None) -> str | None:
    """Provides for example 'gap-4' for 16px, 'p-6' for 24px"""
    if px is None or px == 0:
        return None

    px_int = int(round(px))
    if px_int in TAILWIND_SPACING:
        return f'{prefix}-{TAILWIND_SPACING[px_int]}'

    # Fallback: arbitrary value
    return f'{prefix}-[{px_int}px]'

In [29]:
def frame_to_classes(node: dict) -> list[str]:
    """Converts FRAME-layout-properties to Tailwind classes"""
    classes = []
    layout_mode = node.get('layoutMode')

    if layout_mode == 'HORIZONTAL':
        classes.append('flex')
    elif layout_mode == 'VERTICAL':
        classes.append('flex flex-col')

    # Spacing
    gap = _spacing_class('gap', node.get('itemSpacing'))
    if gap:
        classes.append(gap)

    # Padding — Simplify symmetrically if possible
    pl, pr = node.get('paddingLeft'), node.get('paddingRight')
    pt, pb = node.get('paddingTop'), node.get('paddingBottom')
    if pl == pr == pt == pb and pl:
        c = _spacing_class('p', pl)

        if c: classes.append(c)
    else:
        if pl == pr and pl:
            c = _spacing_class('px', pl)

            if c: classes.append(c)
        else:
            for px, prefix in [(pl, 'pl'), (pr, 'pr')]:
                c = _spacing_class(prefix, px)

                if c: classes.append(c)
        if pt == pb and pt:
            c = _spacing_class('py', pt)

            if c: classes.append(c)
        else:
            for px, prefix in [(pt, 'pt'), (pb, 'pb')]:
                c = _spacing_class(prefix, px)

                if c: classes.append(c)

    # Cross-Axis Alignment
    counter = node.get('counterAxisAlignItems')
    if counter == 'CENTER':
        classes.append('items-center')
    elif counter == 'MAX':
        classes.append('items-end')

    # Primary-Axis Alignment
    primary = node.get('primaryAxisAlignItems')
    if primary == 'CENTER':
        classes.append('justify-center')
    elif primary == 'MAX':
        classes.append('justify-end')
    elif primary == 'SPACE_BETWEEN':
        classes.append('justify-between')

    return classes

## 5. AST-Builder: Figma-JSON → UINode


In [30]:
def _normalize_name(name: str) -> str:
    """'Input Text' / 'input-text' / 'InputText' -> 'inputtext'"""
    return re.sub(r'[\s\-_]+', '', name or '').lower()


def _find_child_by_name(figma_node: dict, target_name: str) -> dict | None:
    for c in figma_node.get('children', []) or []:
        if _normalize_name(c.get('name', '')) == _normalize_name(target_name):
            return c

    return None


def _find_descendant_by_name(figma_node: dict, target_name: str) -> dict | None:
    """Recursive search throughout the entire subtree"""
    target = _normalize_name(target_name)

    for c in figma_node.get('children', []) or []:
        if _normalize_name(c.get('name', '')) == target:
            return c

        found = _find_descendant_by_name(c, target_name)

        if found is not None:
            return found

    return None

### 5.1 Instance-Strategy: Name-based Mapping of PrimeVue-Components

In [31]:
def _transform_instance(node: dict) -> UINode | None:
    """A Figma instance is mapped to a PrimeVue component"""
    name_key = _normalize_name(node.get('name', ''))

    if name_key in ICON_SKIP_INSTANCES:
        return None

    spec = COMPONENT_MAP.get(name_key)

    if not spec:
        # Fallback if instance in defined as FRAME
        if name_key in FRAME_MAP:
            return _transform_frame(node)

         # Unknown Component — Display as a generic div with a comment
        _metrics['instances_unmapped'] += 1

        return UINode(
            tag='div',
            classes=['border', 'border-dashed', 'border-red-400', 'p-2'],
            children=[f'<!-- Unmapped INSTANCE: {node.get("name")!r} -->'],
            figma_id=node.get('id'),
            figma_name=node.get('name'),
        )

    _metrics['instances_mapped'] += 1

    figma_props = node.get('componentProperties', {}) or {}
    static, dynamic = apply_property_rules(figma_props, spec)

    # extra_attrs (e.g. example v-model)
    extra = spec.get('extra_attrs', {})
    nid = 'n' + (node.get('id') or '').replace(':', '_').replace(';', '_').replace('-', '_')
    for k, v in extra.items():
        if isinstance(v, str) and '{node_id}' in v:
            v = v.replace('{node_id}', nid)

        if isinstance(v, bool):
            if v:
                static[k] = ''  # # Boolean attribute without a value
        else:
            static[k] = v

    # Inherit placeholder from a sub-instance (e.g. password.placeholder aus _inputtext-content)
    sub_name = spec.get('inherit_placeholder_from_sub')
    if sub_name and 'placeholder' not in static:
        sub = _find_descendant_by_name(node, sub_name)

        if sub:
            sub_props = sub.get('componentProperties', {}) or {}
            text_config = _extract_value(sub_props.get('Text Config'))

            if text_config == 'Placeholder':
                ph = _extract_value(sub_props.get('Placeholder#4275:140'))

                if ph:
                    static['placeholder'] = str(ph)
            else:
                val = _extract_value(sub_props.get('Value#4275:146'))

                if val:
                    static['value'] = str(val)

    # Slot-Strategy
    children = []
    if spec.get('slot_strategy') == 'default':
        for c in node.get('children', []) or []:
            sub = transform_node(c)

            if sub:
                children.append(sub)

    # Avatar with OverlayBadge
    if spec.get('overlay_badge_child'):
        show_badge = _extract_value(figma_props.get('Show Badge#2138:0', {})) == True

        if show_badge:
            badge_node = _find_child_by_name(node, 'overlaybadge')

            if badge_node:
                badge_ui = transform_node(badge_node)

                if badge_ui:
                    # Avatar becomes the default slot for OverlayBadge
                    avatar_ui = UINode(
                        tag=spec['primevue'],
                        props=static,
                        dynamic_props=dynamic,
                        self_closing=True,
                        is_component=True,
                        figma_id=node.get('id'),
                        figma_name=node.get('name'),
                    )

                    badge_ui.children = [avatar_ui]
                    badge_ui.self_closing = False

                    return badge_ui

    # Post-Processing: Converting a percentage string to a numerical value for a progress bar
    pct_key = spec.get('parse_value_from')
    if pct_key and pct_key in figma_props:
        pct_text = str(_extract_value(figma_props[pct_key])).replace('%', '').strip()

        try:
            dynamic['value'] = str(int(float(pct_text)))
        except ValueError:
            pass

    static.pop('_pct_raw', None)

    # Post-Processing: Icon Only → label-Prop unterdrücken
    icon_only_key = spec.get('icon_only_prop')
    if icon_only_key:
        raw_io = _extract_value(figma_props.get(icon_only_key, {}))

        if str(raw_io).lower() == 'true':
            static.pop('label', None)

    return UINode(
        tag=spec['primevue'],
        props=static,
        dynamic_props=dynamic,
        children=children,
        is_component=True,
        self_closing=not children,
        figma_id=node.get('id'),
        figma_name=node.get('name'),
    )

### 5.2 Frame Strategy: Name-based mapping to PrimeVue components with specific layout rules (slots)

In [32]:
def _find_descendants(node: dict, name: str) -> list[dict]:
    """Find all descendants with normalized names (recursively)"""
    target = _normalize_name(name)
    results = []
    for child in node.get('children', []) or []:
        if not isinstance(child, dict): continue
        if _normalize_name(child.get('name','')) == target:
            results.append(child)
        results.extend(_find_descendants(child, name))
    return results


# ------- Card -------
def _strategy_card(figma_node: dict, spec: dict) -> UINode:
    """Card: body-Frame transparent; header/content/footer → named slots"""
    slot_map = spec.get('slot_map', {})
    children = []

    def process(frame):
        for child in frame.get('children', []) or []:
            if not isinstance(child, dict): continue
            cname = _normalize_name(child.get('name', ''))

            if cname == 'body':
                process(child)  # body-Frame: pass through transparently
            elif cname in slot_map:
                sub = transform_node(child)

                if sub:
                     # Unwrap the wrapper div of a slot frame:
                    # If `transform_node` produces a pure layout div,
                    # move its classes directly into the child list
                    if (isinstance(sub, UINode) and sub.tag == 'div'
                            and not sub.is_component and not sub.props):
                        for inner_child in sub.children:
                            if isinstance(inner_child, UINode):
                                inner_child.slot = slot_map[cname]
                            children.append(inner_child)
                    else:
                        sub.slot = slot_map[cname]
                        children.append(sub)

            else:
                # caption etc. → Default-Slot
                sub = transform_node(child)

                if sub: children.append(sub)

    process(figma_node)

    return UINode(
        tag='Card',
        is_component=True,
        children=children,
        self_closing=not children,
        figma_id=figma_node.get('id'),
        figma_name=figma_node.get('name'),
    )


# ------- Generic Slot Map (Dialog, etc.) -------
def _strategy_slot_map(figma_node: dict, spec: dict) -> UINode:
    """Slot map strategy for dialogs, popover content, etc"""
    slot_map = spec.get('slot_map', {})
    children = []
    static, dynamic = {}, {}

    nid = 'n' + figma_node.get('id','').replace(':','_').replace(';','_').replace('-','_')

    for k, v in spec.get('extra_attrs', {}).items():
        v = v.replace('{node_id}', nid) if isinstance(v, str) else v

        if k.startswith(':'):
            dynamic[k[1:]] = str(v)
        else:
            static[k] = str(v)

    for child in figma_node.get('children', []) or []:
        if not isinstance(child, dict): continue

        cname = _normalize_name(child.get('name', ''))
        slot_target = slot_map.get(cname)
        sub = transform_node(child)

        if sub is None: continue

        if slot_target:
            sub.slot = slot_target

        children.append(sub)

    return UINode(
        tag=spec['primevue'],
        is_component=True,
        props=static,
        dynamic_props=dynamic,
        children=children,
        self_closing=not children,
        figma_id=figma_node.get('id'),
        figma_name=figma_node.get('name'),
    )


# ------- Tabs -------
def _strategy_tabs(figma_node: dict, spec: dict) -> UINode:
    """Tabs → <Tabs><TabList><Tab>...</Tab></TabList><TabPanels>...</TabPanels></Tabs>"""
    tab_headers, tab_panels = [], []

    for child in figma_node.get('children', []) or []:
        if not isinstance(child, dict): continue

        cname = _normalize_name(child.get('name', ''))

        if cname == 'tablist':
            for item in child.get('children', []) or []:
                if not isinstance(item, dict): continue

                if item.get('type') == 'INSTANCE' and '_tabs-tab' in item.get('name',''):
                    props = item.get('componentProperties', {}) or {}
                    header = _extract_value(props.get('Header#4272:96', {}))

                    if header: tab_headers.append(str(header))

        elif 'tabpanel' in cname or cname.startswith('_tabs'):
            # tabpanels container → extract tabpanel frames
            for panel in child.get('children', []) or []:
                if not isinstance(panel, dict): continue

                if 'tabpanel' in _normalize_name(panel.get('name', '')):
                    panel_children = [transform_node(c) for c in panel.get('children', []) or []
                                      if isinstance(c, dict)]
                    tab_panels.append([c for c in panel_children if c])

    # Build the TabList
    tab_list = UINode(
        tag='TabList',
        is_component=True,
        self_closing=False,
        children=[
            UINode(
                tag='Tab',
                is_component=True,
                props={'value': str(i)},
                children=[h],
                self_closing=False
            )
            for i, h in enumerate(tab_headers)
        ],
    )

    # Creating TabPanels
    panels_node = UINode(
        tag='TabPanels',
        is_component=True,
        self_closing=False,
        children=[
            UINode(
                tag='TabPanel',
                is_component=True,
                props={'value': str(i)},
                children=children,
                self_closing=not children
            )
             for i, children in enumerate(tab_panels)
        ],
    )

    return UINode(
        tag='Tabs',
        is_component=True,
        props={'value': '0'},
        children=[tab_list, panels_node],
        self_closing=False,
        figma_id=figma_node.get('id'),
        figma_name=figma_node.get('name'),
    )


# ------- Select -------
def _strategy_select(figma_node: dict, spec: dict) -> UINode:
    """Select → extracts the label from _select-input and the options from _select-option"""
    label, placeholder, disabled = '', '', False

    # Label + Props aus _select-input
    for child in figma_node.get('children', []) or []:
        if not isinstance(child, dict): continue

        if '_select-input' in child.get('name', ''):
            cp = child.get('componentProperties', {}) or {}
            label       = str(_extract_value(cp.get('↳ Label#5662:83', {})) or '')
            placeholder = str(_extract_value(cp.get('↳ Float Label#4275:152', {})) or '')
            disabled    = str(_extract_value(cp.get('🚫 Disabled', {})) or 'False') == 'True'

    # Options from _select-option descendants
    options = []
    for opt in _find_descendants(figma_node, '_select-option'):
        cp = opt.get('componentProperties', {}) or {}
        text = str(_extract_value(cp.get('Text#4000:0', {})) or '')

        if text: options.append(text)

    # Wrapper with label (if available) + Select
    nid = 'n' + figma_node.get('id','').replace(':','_').replace(';','_').replace('-','_')
    options_var = f'_options_{nid}'

    select_node = UINode(
        tag='Select',
        is_component=True,
        props={
            'v-model': f'_state.{nid}',
            ':options': options_var,
            **({'placeholder': placeholder} if placeholder else {}),
        },
        dynamic_props={'disabled': 'true'} if disabled else {},
        self_closing=True,
        figma_id=figma_node.get('id'),
        figma_name=figma_node.get('name'),
    )

    if label:
        return UINode(
            tag='div',
            classes=['flex', 'flex-col', 'gap-2'],
            children=[
                UINode(
                    tag='label',
                    children=[label]
                ),
                select_node,
            ],
        )

    return select_node


# ------- DataTable -------
def _strategy_datatable(figma_node: dict, spec: dict) -> UINode:
    """DataTable → Columns from the header, cell templates from the first row"""
    headers, cell_templates = [], []

    thead = next((c for c in figma_node.get('children', []) or []
                  if _normalize_name(c.get('name','')) == 'thead'), None)
    tbody = next((c for c in figma_node.get('children', []) or []
                  if _normalize_name(c.get('name','')) == 'tbody'), None)

    if thead:
        for cell in thead.get('children', []) or []:
            if not isinstance(cell, dict): continue

            cp = cell.get('componentProperties', {}) or {}
            h = str(_extract_value(cp.get('Header#4295:25', {})) or cell.get('name', ''))

            if h: headers.append(h)

            # Fallback: TEXT child
            if not cp:
                for sub in cell.get('children', []) or []:
                    if isinstance(sub, dict) and sub.get('type') == 'TEXT':
                        headers.append(sub.get('characters', ''))

                        break

    if tbody:
        first_row = next((r for r in tbody.get('children', []) or []), None)
        if first_row:
            for body_cell in first_row.get('children', []) or []:
                if not isinstance(body_cell, dict): continue

                content_frames = _find_descendants(body_cell, '_datatable-content')
                content_frame = content_frames[0] if content_frames else body_cell
                cell_children = [transform_node(c)
                                  for c in content_frame.get('children', []) or []
                                  if isinstance(c, dict)]
                cell_templates.append([c for c in cell_children if c])

    # Columns aufbauen
    columns = []
    for i, header in enumerate(headers):
        body_content = cell_templates[i] if i < len(cell_templates) else []
        body_template = UINode(
            tag='template',
            props={'#body': ''},
            children=body_content
        )
        col = UINode(
            tag='Column',
            is_component=True,
            props={'header': header},
            children=[body_template] if body_content else [],
            self_closing=not body_content,
        )
        columns.append(col)

    return UINode(
        tag='DataTable',
        is_component=True,
        props={':value': 'items'},
        children=columns,
        self_closing=not columns,
        figma_id=figma_node.get('id'),
        figma_name=figma_node.get('name'),
    )


# ------- Popover -------
def _strategy_popover(figma_node: dict, spec: dict) -> UINode:
    """Popover: Expand nested ‘popover’ frames; use content as the default slot"""
    def find_content(node):
        results = []

        for child in node.get('children', []) or []:
            if not isinstance(child, dict): continue
            cname = _normalize_name(child.get('name', ''))

            # Nested popover wrappers → resolve recursively
            if cname == 'popover' and child.get('type') == 'FRAME':
                results.extend(find_content(child))
            else:
                sub = transform_node(child)
                if sub: results.append(sub)

        return results

    children = find_content(figma_node)
    nid = 'n' + figma_node.get('id','').replace(':','_').replace(';','_').replace('-','_')

    return UINode(
        tag='Popover',
        is_component=True,
        props={'ref': f'op_{nid}'},
        children=children,
        self_closing=not children,
        figma_id=figma_node.get('id'),
        figma_name=figma_node.get('name'),
    )


# ------- Breadcrumb -------
def _strategy_breadcrumb(figma_node: dict, spec: dict) -> UINode:
    """Breadcrumb → an array of _breadcrumb-item instances"""
    items = []
    for child in figma_node.get('children', []) or []:
        if not isinstance(child, dict): continue
        if '_breadcrumb-item' not in child.get('name', ''): continue

        cp = child.get('componentProperties', {}) or {}
        item_type = str(_extract_value(cp.get('Type', {})) or 'Label')

        if item_type == 'Icon':
            items.append("{ icon: 'pi pi-home' }")
        else:
            #Label text from TEXT-child
            text = next(
                (c.get('characters','') for c in child.get('children',[]) or []
                 if isinstance(c,dict) and c.get('type')=='TEXT'), 'Item'
            )
            items.append(f"{{ label: '{text}' }}")

    items_str = '[' + ', '.join(items) + ']' if items else '[]'
    nid = 'n' + figma_node.get('id','').replace(':','_').replace(';','_').replace('-','_')

    return UINode(
        tag='Breadcrumb',
        is_component=True,
        props={':model': f'_breadcrumb_{nid}'},
        self_closing=True,
        figma_id=figma_node.get('id'),
        figma_name=figma_node.get('name'),
    )


# ------- Accordion -------
def _strategy_accordion(figma_node: dict, spec: dict) -> UINode:
    """Accordion → AccordionPanel with header + content from _accordion-panel"""
    panels = []
    for i, child in enumerate(figma_node.get('children', []) or []):
        if not isinstance(child, dict): continue
        if '_accordion-panel' not in child.get('name', ''): continue

        cp = child.get('componentProperties', {}) or {}
        toggle = str(_extract_value(cp.get('Toggle Status', {})) or 'Collapsed')

        # Header from _accordion-header descendant
        header_text = ''
        for desc in _find_descendants(child, '_accordion-header'):
            hcp = desc.get('componentProperties', {}) or {}
            header_text = str(_extract_value(hcp.get('Header#4272:33', {})) or '')

            break

        # Content: ‘content’ frame within the panel
        panel_content = []
        for frame in child.get('children', []) or []:
            if isinstance(frame, dict) and _normalize_name(frame.get('name','')) == 'content':
                for sub in frame.get('children', []) or []:
                    sub_node = transform_node(sub)
                    if sub_node: panel_content.append(sub_node)

        acc_header  = UINode(
            tag='AccordionHeader',
            is_component=True,
            children=[header_text],
            self_closing=False
        )
        acc_content = UINode(
            tag='AccordionContent',
            is_component=True,
            children=panel_content,
            self_closing=not panel_content
        )

        panel = UINode(
            tag='AccordionPanel',
            is_component=True,
            props={'value': str(i)},
            children=[acc_header, acc_content],
            self_closing=False,
        )

        panels.append(panel)

    return UINode(
        tag='Accordion',
        is_component=True,
        props={'value': '0'},
        children=panels,
        self_closing=not panels,
        figma_id=figma_node.get('id'),
        figma_name=figma_node.get('name'),
    )

In [33]:
STRATEGY_DISPATCH = {
    'card':      _strategy_card,
    'slot_map':  _strategy_slot_map,
    'tabs':      _strategy_tabs,
    'select':    _strategy_select,
    'datatable': _strategy_datatable,
    'popover':   _strategy_popover,
    'breadcrumb':_strategy_breadcrumb,
    'accordion': _strategy_accordion,
}


In [34]:
def _transform_frame(node: dict) -> UINode | None:
    """FRAME → PrimeVue compound component (via FRAME_MAP) or <div> (fallback)"""
    raw_name = node.get('name', '')
    norm = _normalize_name(raw_name)

    # 1. Transparent wrapper frames (screen, etc.) → return the first child node
    if norm in TRANSPARENT_FRAMES:
        for child in node.get('children', []) or []:
            result = transform_node(child)
            if result: return result

        return None

    # 2. FRAME_MAP-Lookup → Compound-Komponente
    spec = FRAME_MAP.get(norm)
    if spec:
        strategy_fn = STRATEGY_DISPATCH.get(spec.get('strategy', ''))
        if strategy_fn:
            _metrics['frames_compound'] += 1

            return strategy_fn(node, spec)

    # 3. Fallback: generic <div> with Tailwind layout
    classes = frame_to_classes(node)
    children = []
    for c in node.get('children', []) or []:
        sub = transform_node(c)
        if sub: children.append(sub)

    # Individual child without a class of their own → pass them through transparently
    if not classes and len(children) == 1:
        return children[0]

    _metrics['frames_fallback'] += 1

    return UINode(
        tag='div',
        classes=classes,
        children=children,
        figma_id=node.get('id'),
        figma_name=node.get('name'),
    )

In [35]:
def _transform_text(node: dict) -> UINode | None:
    """TEXT node — if the text is NOT property-bound, render it as <span>"""
    if node.get('componentPropertyReferences', {}).get('characters'):
        # Text is set by the parent component via a prop → no separate node
        return None

    text = node.get('characters', '')

    if not text:
        return None

    # Font size mapping (simplified)
    style = node.get('style', {})
    classes = []
    fs = style.get('fontSize')

    if fs:
        size_map = {12: 'text-xs', 14: 'text-sm', 16: 'text-base',
                    18: 'text-lg', 20: 'text-xl', 24: 'text-2xl', 28: 'text-3xl'}

        cls = size_map.get(int(fs))

        if cls:
            classes.append(cls)

    fw = style.get('fontWeight', 400)

    if fw and int(fw) >= 600:
        classes.append('font-semibold' if int(fw) < 700 else 'font-bold')

    return UINode(
        tag='span',
        classes=classes,
        children=[text],
        figma_id=node.get('id'),
        figma_name=node.get('name')
    )

In [36]:
def transform_node(figma_node: dict) -> UINode | None:
    """Converts a single Figma node into a UINode

    Returns None if the node is to be discarded (e.g., internal sub-instances)
    """
    if not isinstance(figma_node, dict):
        return None

    node_type = figma_node.get('type')
    raw_name = figma_node.get('name', '')

    # Internal Helper-Instances (Prefix '_') are dropped
    if raw_name.startswith('_'):
        return None

    if node_type == 'INSTANCE':
        return _transform_instance(figma_node)
    elif node_type == 'FRAME':
        return _transform_frame(figma_node)
    elif node_type == 'TEXT':
        return _transform_text(figma_node)
    else:
        # VECTOR, RECTANGLE, etc. – no treatment at this time
        return None

## 6. Code-Generator: UINode → Vue 3 SFC

In [37]:
def render_ast(node: UINode, depth: int = 0, imports: set | None = None) -> str:
    if imports is None:
        imports = set()

    indent = '  ' * depth

    if isinstance(node, str):
        return f'{indent}{node}'

    if node.is_component:
        imports.add(node.tag)

    # Assembling attributes
    attrs = []

    if node.classes:
        attrs.append(f'class="{" ".join(node.classes)}"')

    for k, v in node.props.items():
        if v == '':
            attrs.append(k)  # Boolean attribute without a value
        else:
            attrs.append(f'{k}="{v}"')

    for k, v in node.dynamic_props.items():
        attrs.append(f':{k}="{v}"')

    attr_str = ''

    if attrs:
        # If there are more than 2 attributes: Multi-line
        if len(attrs) > 2 or sum(len(a) for a in attrs) > 60:
            attr_str = '\n' + '\n'.join(f'{indent}  {a}' for a in attrs) + f'\n{indent}'
        else:
            attr_str = ' ' + ' '.join(attrs)

    if node.self_closing or not node.children:
        return f'{indent}<{node.tag}{attr_str} />'

    # With children
    opening = f'{indent}<{node.tag}{attr_str}>'
    closing = f'{indent}</{node.tag}>'

    rendered_children = []

    for c in node.children:
        if isinstance(c, str):
            # String-Child inline
            rendered_children.append(f'{indent}  {c}')
        elif isinstance(c, UINode) and c.slot:
            # Named Slot → <template #slotname>...</template>
            inner = render_ast(c, depth + 2, imports)
            rendered_children.append(
                f'{indent}  <template #{c.slot}>\n{inner}\n{indent}  </template>'
            )
        else:
            rendered_children.append(render_ast(c, depth + 1, imports))

    # Single string child → inline
    if len(node.children) == 1 and isinstance(node.children[0], str):
        return f'{indent}<{node.tag}{attr_str}>{node.children[0]}</{node.tag}>'

    return opening + '\n' + '\n'.join(rendered_children) + '\n' + closing


def generate_sfc(figma_root: dict) -> str:
    """Generates a complete Vue 3 SFC from a Figma mockup"""
    _reset_metrics()

    ast = transform_node(figma_root)

    if ast is None:
        return '<!-- Unable to transform the mockup -->'

    # Remove the outer default canvas wrapper from generated templates
    if (
        isinstance(ast, UINode)
        and ast.tag == 'div'
        and not ast.is_component
        and ast.classes == ['flex flex-col', 'p-6']
        and not ast.props
        and not ast.dynamic_props
        and len(ast.children) == 1
        and isinstance(ast.children[0], UINode)
    ):
        ast = ast.children[0]

    imports: set[str] = set()
    template_body = render_ast(ast, depth=1, imports=imports)

    # Collect state bindings (for v-model placeholders)
    state_refs = []

    def collect_refs(n):
        if isinstance(n, str): return

        for k, v in n.props.items():
            if k.startswith('v-model') and isinstance(v, str) and v.startswith('_state.'):
                state_refs.append(v.split('.', 1)[1])

        for c in n.children:
            collect_refs(c)

    collect_refs(ast)

    # Script-Setup
    script_lines = ['<script setup>']
    if state_refs:
        script_lines.append("import { reactive } from 'vue'")

    for imp in sorted(imports):
        script_lines.append(f"import {imp} from 'primevue/{imp.lower()}'")

    if state_refs:
        script_lines.append('')
        init = ', '.join(f'{r}: null' for r in state_refs)
        script_lines.append(f'const _state = reactive({{ {init} }})')

    script_lines.append('</script>')

    template = f'<template>\n{template_body}\n</template>'

    _metrics['pv_components_unique'] = len(imports)
    _metrics['state_refs_count'] = len(state_refs)
    _metrics['ast_depth_max'] = _ast_depth(ast)

    return template + '\n\n' + '\n'.join(script_lines) + '\n'

## 7. Apply the pipeline to all mockups

In [38]:
INPUT_DIR_PATH = Path(INPUT_DIR)
OUTPUT_DIR_PATH = Path(OUTPUT_DIR)

input_files = sorted(f for f in INPUT_DIR_PATH.rglob('*.json')
                     if f.parent != INPUT_DIR_PATH)
print(f'Input-Files: {len(input_files)}\n')

results = []
current_level = None

for path in input_files:
    level = path.parent.name

    if level != current_level:
        print(f'\n— {level.upper()} —')

        current_level = level

    with open(path, 'r', encoding='utf-8') as f:
        figma = json.load(f)

    t_start = time.perf_counter()
    sfc = generate_sfc(figma)
    duration_ms = (time.perf_counter() - t_start) * 1000

    m = dict(_metrics)
    total_inst = m['instances_mapped'] + m['instances_unmapped']
    m['instances_total'] = total_inst
    m['mapping_coverage'] = round(
        m['instances_mapped'] / total_inst, 4
    ) if total_inst else 1.0

    rel = path.relative_to(INPUT_DIR).with_stem(path.stem + '-a').with_suffix('.vue')
    out_path = OUTPUT_DIR_PATH / rel
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(sfc)

    results.append({                                        # ← Statt Tupel: Dict
        'input': path.name,
        'output': out_path.name,
        'complexity': path.parent.name,
        'duration_ms': round(duration_ms, 4),
        'sfc_bytes': len(sfc),
        'sfc_lines': sfc.count('\n') + 1,
        **m,
    })

    print(f'  {path.name:40s} -> {out_path.name:40s}  ({len(sfc):4d} bytes, {duration_ms:6.2f} ms)')

total_ms = sum(r['duration_ms'] for r in results)
print(f'\nGenerated: {len(results)} SFCs — Total: {total_ms:.2f} ms — Avg: {total_ms / len(results):.2f} ms')

Input-Files: 30


— HARD —
  1.json                                   -> 1-a.vue                                   (1502 bytes,   0.68 ms)
  10.json                                  -> 10-a.vue                                  (1950 bytes,   0.60 ms)
  2.json                                   -> 2-a.vue                                   ( 901 bytes,   0.35 ms)
  3.json                                   -> 3-a.vue                                   ( 929 bytes,   0.43 ms)
  4.json                                   -> 4-a.vue                                   (1379 bytes,   0.45 ms)
  5.json                                   -> 5-a.vue                                   (2104 bytes,   0.54 ms)
  6.json                                   -> 6-a.vue                                   ( 418 bytes,   0.26 ms)
  7.json                                   -> 7-a.vue                                   (2862 bytes,   0.82 ms)
  8.json                                   -> 8-a.vue                        

In [39]:
# Group by complexity level
by_complexity = defaultdict(list)
for r in results:
    by_complexity[r['complexity']].append(r)

def _avg(values):
    return round(sum(values) / len(values), 4) if values else 0

per_level = {}
for level, items in by_complexity.items():
    per_level[level] = {
        'count':            len(items),
        'avg_duration_ms':  _avg([i['duration_ms']      for i in items]),
        'avg_coverage':     _avg([i['mapping_coverage'] for i in items]),
        'avg_depth':        _avg([i['ast_depth_max']    for i in items]),
        'total_unmapped':   sum(i['instances_unmapped'] for i in items),
        'total_fallback':   sum(i['frames_fallback']    for i in items),
    }

metrics_report = {
    'method': 'A',
    'summary': {
        'total_files': len(results),
        'total_ms': round(sum(r['duration_ms'] for r in results), 4),
        'avg_ms':   round(sum(r['duration_ms'] for r in results) / len(results), 4),
        'avg_coverage':   round(sum(r['mapping_coverage'] for r in results) / len(results), 4),
        'total_unmapped': sum(r['instances_unmapped'] for r in results),
        'total_fallback': sum(r['frames_fallback'] for r in results),
    },
    'files': results,
}

report_path = Path('reports') / 'metrics_report_a.json'
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(metrics_report, f, indent=4, ensure_ascii=False)

    print(f'Transformation report saved to: {report_path}')

Transformation report saved to: reports\metrics_report_a.json


## 8. Coverage-Diagnose

In [40]:
mapped = Counter()
unmapped = Counter()

def diagnose(figma_node, inside_dropped: bool = False):
    if not isinstance(figma_node, dict):
        return

    if figma_node.get('type') == 'INSTANCE':
        name = figma_node.get('name', '')

        if name.startswith('_'):
            return  # Internal sub-instance, intentionally dropped

        if inside_dropped:
            return  # Children of a drop-strategy component are intentionally ignored

        key = _normalize_name(name)

        if key in COMPONENT_MAP:
            mapped[name] += 1
            spec = COMPONENT_MAP[key]

            # If slot_strategy=‘drop’: do not descend into Children
            if spec.get('slot_strategy') == 'drop' and not spec.get('overlay_badge_child'):
                return
        else:
            unmapped[name] += 1

    for c in figma_node.get('children', []) or []:
        diagnose(c, inside_dropped)

for path in input_files:
    with open(path, 'r', encoding='utf-8') as f:
        diagnose(json.load(f))

print(f'Mapped instances:   {sum(mapped.values()):4d}')
for n, c in mapped.most_common():
    print(f'  {n:25s} {c:3d}x')

print(f'\nUnmapped Instances: {sum(unmapped.values()):4d}')
for n, c in unmapped.most_common():
    print(f'  {n:25s} {c:3d}x  <- Add mapping to COMPONENT_MAP')

total = sum(mapped.values()) + sum(unmapped.values())
if total:
    print(f'\nCoverage: {sum(mapped.values()) / total * 100:.1f}%')

Mapped instances:    162
  button                     49x
  tag                        21x
  avatar                     15x
  checkbox                   13x
  divider                    13x
  radiobutton                13x
  inputtext                   6x
  progressbar                 6x
  inputnumber                 4x
  toggleswitch                4x
  skeleton                    4x
  datepicker                  3x
  textarea                    3x
  password                    3x
  slider                      3x
  menu                        1x
  overlaybadge                1x

Unmapped Instances:    8
  chevron-right               5x  <- Add mapping to COMPONENT_MAP
  user-edit                   1x  <- Add mapping to COMPONENT_MAP
  shield                      1x  <- Add mapping to COMPONENT_MAP
  tabs                        1x  <- Add mapping to COMPONENT_MAP

Coverage: 95.3%
